# 01 — Deterministic energy, modes, and mixed-mode algebra

Companion to the deterministic theory. It derives the energy balance, modal coefficients, energy-normalized geometry, and the full mixed-mode energy expression used to interpret general preparations.


## Guiding question
How does an initial electrical state split into slow and fast modes, and how do those modal weights enter the stored energy?


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, solve_continuous_lyapunov
np.set_printoptions(precision=6, suppress=True)


## 1. Energy balance
With $G=\mathrm{diag}(1/C,L)$,
$$
E(X)=\frac12X^TGX=\frac{q^2}{2C}+\frac{LI^2}{2}.
$$
Using $\dot q=I$ and $L\dot I+RI+q/C=0$ gives $\dot E=-RI^2\le0$. Individual trajectories are monotone even when the ordering of two trajectories reverses.


In [2]:
L=44.4e-3; C=454e-6; R=27.9
A=np.array([[0.,1.],[-1/(L*C),-R/L]])
G=np.diag([1/C,L])
w,V=np.linalg.eig(A)
idx=np.argsort(w)[::-1]
w=w[idx].real; V=V[:,idx].real
lam_s,lam_f=w
rs=np.array([1.,lam_s]); rf=np.array([1.,lam_f])
print('rates:',lam_s,lam_f)
assert np.allclose(A@rs,lam_s*rs)
assert np.allclose(A@rf,lam_f*rf)


rates: -92.59098035781062 -535.7873980205677


## 2. Modal coefficients
Choose eigenvectors $r_j=(1,\lambda_j)^T$. If $X_0=c_sr_s+c_fr_f$, then
$$
c_s=\frac{I_0-\lambda_f q_0}{\lambda_s-\lambda_f},\qquad
c_f=\frac{\lambda_s q_0-I_0}{\lambda_s-\lambda_f}.
$$
Pure slow and fast preparations satisfy $I_0=\lambda_s q_0$ and $I_0=\lambda_f q_0$.


In [3]:
def modal_coeffs(q0,I0,ls,lf):
    cs=(I0-lf*q0)/(ls-lf)
    cf=(ls*q0-I0)/(ls-lf)
    return cs,cf
q0=1e-6
assert np.allclose(modal_coeffs(q0,lam_s*q0,lam_s,lam_f),(q0,0))
assert np.allclose(modal_coeffs(q0,lam_f*q0,lam_s,lam_f),(0,q0))
print('pure-mode coefficient checks: PASS')


pure-mode coefficient checks: PASS


## 3. Energy-normalized coordinates
Define
$$
z_1=q/\sqrt C,\qquad z_2=\sqrt L\,I.
$$
Then $E=(z_1^2+z_2^2)/2$: energy is radius, while direction controls modal composition.


In [4]:
S=np.diag([1/np.sqrt(C),np.sqrt(L)])
Sinv=np.linalg.inv(S)
assert np.allclose(Sinv.T@S.T@S@Sinv,np.eye(2))
# numerical energy identity
x=np.array([2.2e-5,-4e-4]); z=S@x
assert np.isclose(0.5*x@G@x,0.5*z@z)
print('energy-normalized identity: PASS')


energy-normalized identity: PASS



## 4. Modal energy metric and nonorthogonality

Collect the slow and fast eigenvectors into

$$
V=(r_s,r_f),
\qquad
r_s=
\begin{pmatrix}
1\\
\lambda_s
\end{pmatrix},
\qquad
r_f=
\begin{pmatrix}
1\\
\lambda_f
\end{pmatrix}.
$$

The energy metric in the modal basis is

$$
V^TGV=
\begin{pmatrix}
r_s^TGr_s & r_s^TGr_f\\
r_f^TGr_s & r_f^TGr_f
\end{pmatrix}
=
\begin{pmatrix}
\displaystyle \frac{1}{C}+L\lambda_s^2 &
\displaystyle \frac{1}{C}+L\lambda_s\lambda_f\\
\displaystyle \frac{1}{C}+L\lambda_s\lambda_f &
\displaystyle \frac{1}{C}+L\lambda_f^2
\end{pmatrix}.
$$

For the series RLC characteristic equation,

$$
\lambda^2+\frac{R}{L}\lambda+\frac{1}{LC}=0,
$$

the product of the two roots is

$$
\lambda_s\lambda_f=\frac{1}{LC}.
$$

Therefore the off-diagonal energy overlap simplifies to

$$
r_s^TGr_f
=
\frac{1}{C}+L\lambda_s\lambda_f
=
\frac{2}{C}.
$$

The slow and fast eigenvectors are therefore not orthogonal in the physical energy metric.  This is why a general mixed preparation contains the cross term proportional to
$e^{(\lambda_s+\lambda_f)t}$ even though the dynamical matrix itself is diagonal in the modal basis.


In [5]:

Vmodal = np.column_stack([rs, rf])
Gmodal = Vmodal.T @ G @ Vmodal

Gmodal_analytic = np.array([
    [1/C + L*lam_s**2, 1/C + L*lam_s*lam_f],
    [1/C + L*lam_s*lam_f, 1/C + L*lam_f**2],
])

assert np.allclose(Gmodal, Gmodal_analytic)
assert np.isclose(lam_s*lam_f, 1/(L*C))
assert np.isclose(rs @ G @ rf, 2/C)

print("V^T G V =")
print(Gmodal)
print("r_s^T G r_f =", rs @ G @ rf)
print("2/C           =", 2/C)
print("modal-metric identities: PASS")


V^T G V =
[[ 2583.288352  4405.286344]
 [ 4405.286344 14948.468405]]
r_s^T G r_f = 4405.286343612335
2/C           = 4405.286343612335
modal-metric identities: PASS


## 5. Full mixed-mode energy
For
$$
X(t)=c_sr_se^{\lambda_st}+c_fr_fe^{\lambda_ft},
$$
the quadratic energy is
$$
E(t)=\frac12c_s^2(r_s^TGr_s)e^{2\lambda_st}
+\frac12c_f^2(r_f^TGr_f)e^{2\lambda_ft}
+c_sc_f(r_s^TGr_f)e^{(\lambda_s+\lambda_f)t}.
$$
The third term is the mixed slow--fast sector. It is absent only for a pure-mode preparation or when the relevant metric overlap vanishes.


In [6]:
cs,cf=1.2e-5,-0.7e-5
t=np.linspace(0,0.04,300)
Emix=[]; Edirect=[]
for tt in t:
    X=cs*rs*np.exp(lam_s*tt)+cf*rf*np.exp(lam_f*tt)
    Edirect.append(0.5*X@G@X)
    Emix.append(0.5*cs**2*(rs@G@rs)*np.exp(2*lam_s*tt)
                +0.5*cf**2*(rf@G@rf)*np.exp(2*lam_f*tt)
                +cs*cf*(rs@G@rf)*np.exp((lam_s+lam_f)*tt))
assert np.allclose(Edirect,Emix)
print('mixed-mode expansion: PASS; G-overlap =',rs@G@rf)


mixed-mode expansion: PASS; G-overlap = 4405.286343612335



## 6. Small errors in a nominal fast preparation

Suppose the intended fast preparation has the correct initial charge but a small current error,

$$
I_0=\lambda_f q_0+\delta I.
$$

Substituting this state into the modal coefficients gives

$$
c_s
=
\frac{I_0-\lambda_f q_0}{\lambda_s-\lambda_f}
=
\frac{\delta I}{\lambda_s-\lambda_f}.
$$

The error therefore injects a slow-mode amplitude that is linear in $\delta I$.  At late times the fast contribution and the mixed slow--fast contribution have decayed more rapidly, so the energy is dominated by

$$
E_{\mathrm{tail}}(t)
\simeq
K_{ss}
\left(
\frac{\delta I}{\lambda_s-\lambda_f}
\right)^2
e^{2\lambda_s t},
\qquad
K_{ss}\equiv \frac12 r_s^TGr_s.
$$

Thus the unwanted slow amplitude is first order in the preparation error, whereas its late-time energy contribution is second order.  This distinction is useful experimentally: a visually small misalignment from the fast eigendirection can eventually dominate the tail because the injected slow component decays much more slowly.


In [7]:

Kss = 0.5 * (rs @ G @ rs)

q0 = 20e-6
deltaI_values = np.array([1e-5, 2e-5, 4e-5])
tail_prefactors = []

for deltaI in deltaI_values:
    I0 = lam_f*q0 + deltaI
    cs_err, cf_err = modal_coeffs(q0, I0, lam_s, lam_f)

    # Exact modal identity for the injected slow amplitude.
    assert np.isclose(cs_err, deltaI/(lam_s-lam_f))

    # Coefficient multiplying exp(2 lambda_s t) in the asymptotic slow tail.
    tail_prefactors.append(Kss * cs_err**2)

tail_prefactors = np.array(tail_prefactors)

# Doubling deltaI must multiply the slow-tail energy prefactor by four.
assert np.allclose(tail_prefactors[1:] / tail_prefactors[:-1], 4.0)

print("K_ss =", Kss)
print("delta I values:", deltaI_values)
print("slow-tail prefactors:", tail_prefactors)
print("quadratic preparation-error scaling: PASS")


K_ss = 1291.6441759914583
delta I values: [0.00001 0.00002 0.00004]
slow-tail prefactors: [0. 0. 0.]
quadratic preparation-error scaling: PASS



## 7. Fixed-energy preparation scales

For a fixed initial energy $E_0$,

$$
\frac{q_0^2}{C}+LI_0^2=2E_0.
$$

A convenient parameterization is

$$
q_0(\theta)=\sqrt{2CE_0}\cos\theta,
\qquad
I_0(\theta)=\sqrt{\frac{2E_0}{L}}\sin\theta.
$$

The characteristic preparation scales are therefore

$$
q_{\mathrm{scale}}=\sqrt{2CE_0},
\qquad
I_{\mathrm{scale}}=\sqrt{\frac{2E_0}{L}}.
$$

The pure-mode directions follow from $I_0=\lambda_j q_0$:

$$
\tan\theta_j=\lambda_j\sqrt{LC},
\qquad
j\in\{s,f\}.
$$

Antipodal points on the ellipse correspond to the same modal direction with the opposite overall sign.


In [8]:

E0 = 1e-3
q_scale = np.sqrt(2*C*E0)
I_scale = np.sqrt(2*E0/L)

theta_s = np.arctan(lam_s*np.sqrt(L*C))
theta_f = np.arctan(lam_f*np.sqrt(L*C))

def equal_energy_state(theta):
    q = q_scale*np.cos(theta)
    I = I_scale*np.sin(theta)
    return np.array([q, I])

Xs = equal_energy_state(theta_s)
Xf = equal_energy_state(theta_f)

assert np.isclose(0.5*Xs @ G @ Xs, E0)
assert np.isclose(0.5*Xf @ G @ Xf, E0)
assert np.isclose(Xs[1]/Xs[0], lam_s)
assert np.isclose(Xf[1]/Xf[0], lam_f)

print("q_scale =", q_scale)
print("I_scale =", I_scale)
print("theta_s =", theta_s, "rad")
print("theta_f =", theta_f, "rad")
print("fixed-energy pure-mode geometry: PASS")


q_scale = 0.0009528903399657276
I_scale = 0.21223817998900443
theta_s = -0.3939737543717867 rad
theta_f = -1.1768225724231096 rad
fixed-energy pure-mode geometry: PASS



## 8. Inverse design from a desired rate separation

Let

$$
\rho=\left|\frac{\lambda_f}{\lambda_s}\right|>1
$$

measure the separation between the fast and slow decay rates, and define

$$
\omega_0=\frac{1}{\sqrt{LC}},
\qquad
\gamma=\frac{R}{L}.
$$

Because

$$
\lambda_s\lambda_f=\omega_0^2,
$$

the magnitudes of the two overdamped roots can be written as

$$
|\lambda_s|=\frac{\omega_0}{\sqrt{\rho}},
\qquad
|\lambda_f|=\omega_0\sqrt{\rho}.
$$

Since both roots are negative,

$$
\gamma
=
-(\lambda_s+\lambda_f)
=
\omega_0
\left(
\sqrt{\rho}+\frac{1}{\sqrt{\rho}}
\right).
$$

The required resistance is therefore

$$
R
=
L\gamma
=
\sqrt{\frac{L}{C}}
\left(
\sqrt{\rho}+\frac{1}{\sqrt{\rho}}
\right).
$$

This separates two design roles.  The product $LC$ sets the natural time scale
$\omega_0^{-1}=\sqrt{LC}$, while the ratio $L/C$ sets the impedance scale
$\sqrt{L/C}$ and therefore the resistance required for a chosen modal-rate ratio.


In [9]:

rho = abs(lam_f/lam_s)
omega0 = 1/np.sqrt(L*C)
gamma_from_rho = omega0*(np.sqrt(rho) + 1/np.sqrt(rho))
R_from_rho = np.sqrt(L/C)*(np.sqrt(rho) + 1/np.sqrt(rho))

assert np.isclose(gamma_from_rho, R/L)
assert np.isclose(R_from_rho, R)

print("rho =", rho)
print("gamma reconstructed from rho =", gamma_from_rho)
print("R reconstructed from rho =", R_from_rho)
print("inverse-design identity: PASS")


rho = 5.786604655767323
gamma reconstructed from rho = 628.3783783783784
R reconstructed from rho = 27.900000000000002
inverse-design identity: PASS


## Reader exploration
Choose a fixed energy $E_0$ and move around the preparation ellipse.  For each angle, calculate $(c_s,c_f)$ and identify where one coefficient vanishes.  Then introduce a small current offset around the fast direction and verify numerically that the late-time slow-tail energy scales as $(\delta I)^2.  Finally, vary the target ratio $\rho=|\lambda_f/\lambda_s|$ and use the inverse-design formula to see how the required resistance changes.